# Newton–Krylov · Interactive dashboard

Compare four solvers on a nonlinear benchmark by changing the initial guess and tolerance.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, clear_output

# Locate Newton–Krylov implementation from different launch directories.
candidates = [
    Path.cwd(),
    Path.cwd() / "nonlinear_solvers/newton_krylov",
    Path.cwd() / "newton_krylov",
    Path.cwd().parent / "newton_krylov",
    Path.cwd().parent / "nonlinear_solvers/newton_krylov",
]

package_dir = next(
    (p for p in candidates if (p / "newton_krylov.py").is_file()),
    None
)

if package_dir is None:
    raise FileNotFoundError(
        "Could not locate newton_krylov.py. "
        "Launch the notebook from the repository root or nonlinear_solvers directory."
    )

sys.path.insert(0, str(package_dir.resolve()))

from newton_krylov import (
    newton_gmres,
    jfnk,
    preconditioned_jfnk,
    inexact_jfnk,
)

from newton_krylov import newton_gmres, jfnk, preconditioned_jfnk, inexact_jfnk

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

We solve $F(x_1,x_2)=(x_1\cos(x_2)-4,\;x_2(x_1-1)-5)$.

In [ ]:
def fun(x):
    return np.array([x[0]*np.cos(x[1])-4, x[1]*(x[0]-1)-5])

def jacobian(x):
    return np.array([[np.cos(x[1]), -x[0]*np.sin(x[1])],
                     [x[1], x[0]-1]])

def inverse_diagonal(x, v):
    d = np.diag(jacobian(x))
    return v / np.where(abs(d) < 1e-6, 1.0, d)

METHODS = {
    "Newton–GMRES": lambda x, tol: newton_gmres(fun, x, jacobian=jacobian, tol=tol),
    "JFNK": lambda x, tol: jfnk(fun, x, tol=tol),
    "Right-preconditioned JFNK": lambda x, tol: preconditioned_jfnk(
        fun, x, inverse_preconditioner=inverse_diagonal, tol=tol),
    "Inexact JFNK": lambda x, tol: inexact_jfnk(fun, x, tol=tol),
}

def compute_results(x1=1.0, x2=1.0, tolerance=1e-10):
    start = np.array([float(x1), float(x2)])
    outcomes = {}
    for name, solver in METHODS.items():
        try:
            outcomes[name] = solver(start.copy(), float(tolerance))
        except (ValueError, RuntimeError, FloatingPointError) as exc:
            outcomes[name] = exc
    return outcomes

def render_dashboard(outcomes, tolerance):
    """Show plots first, followed by a collapsible numerical summary."""
    import html
    import ipywidgets as widgets

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
    plotted = False
    for name, outcome in outcomes.items():
        if isinstance(outcome, Exception):
            continue
        plotted = True
        residuals = np.maximum(outcome.residual_norms, np.finfo(float).tiny)
        ax1.semilogy(range(len(residuals)), residuals, '-o', lw=1.7,
                     markersize=4, label=name)
        ax2.plot(range(1, len(outcome.gmres_iterations) + 1),
                 np.cumsum(outcome.gmres_iterations), '-o', lw=1.7,
                 markersize=4, label=name)
    ax1.axhline(tolerance, color='gray', ls='--', lw=1,
                label='Target tolerance')
    ax1.set(title='Nonlinear residual', xlabel='Accepted Newton iteration',
            ylabel=r'$\|F(x_k)\|_2$')
    ax2.set(title='Cumulative inner GMRES iterations',
            xlabel='Newton solve index', ylabel='Cumulative GMRES iterations')
    for ax in (ax1, ax2):
        ax.grid(alpha=.25)
        if plotted:
            ax.legend(fontsize=7)
    plt.show()
    plt.close(fig)

    # Use the actual solver-output structure (outcomes, residual_norms,
    # gmres_iterations), not undefined 'results' or 'inner_iterations'.
    rows = []
    for name, outcome in outcomes.items():
        if isinstance(outcome, Exception):
            status, outer, inner, residual = 'Error', '—', '—', '—'
        else:
            status = 'Converged' if outcome.converged else 'Stopped'
            outer = str(outcome.iterations)
            inner = str(sum(outcome.gmres_iterations))
            residual = f'{outcome.residual_norms[-1]:.3e}'
        cells = [name, status, outer, inner, residual]
        rows.append('<tr>' + ''.join(
            f'<td style="padding:7px 12px;border-bottom:1px solid #e5e7eb">{html.escape(s)}</td>'
            for s in cells) + '</tr>')

    header = ('<tr>' + ''.join(
        f'<th style="text-align:left;padding:8px 12px;border-bottom:2px solid #ddd">{h}</th>'
        for h in ['Method', 'Status', 'Newton', 'GMRES', 'Final ||F||']) + '</tr>')
    note = ('<p style="font-size:13px;color:#555">GMRES counts refer to callbacks, '
            'not wall-clock time.</p>')
    table_html = (f'<p><b>Benchmark:</b> 2 variables &nbsp; | &nbsp; '
                  f'<b>Target residual:</b> ≤ {tolerance:.0e}</p>'
                  '<div style="overflow-x:auto"><table style="border-collapse:collapse;'
                  'font-family:system-ui,sans-serif;font-size:13px">'
                  + header + ''.join(rows) + '</table></div>' + note)
    details = widgets.Accordion(children=[widgets.HTML(value=table_html)])
    details.set_title(0, 'Numerical results — click to expand')
    details.selected_index = None
    display(details)

    failures = [(name, obj.message if not isinstance(obj, Exception) else str(obj))
                for name, obj in outcomes.items()
                if isinstance(obj, Exception) or not obj.converged]
    if failures:
        print('Non-converged runs:')
        for name, message in failures:
            print(f'  • {name}: {message}')


In [ ]:
import ipywidgets as widgets

x1 = widgets.FloatText(value=1.0, description='x₁ start:', layout=widgets.Layout(width='220px'))
x2 = widgets.FloatText(value=1.0, description='x₂ start:', layout=widgets.Layout(width='220px'))
tolerance = widgets.Dropdown(
    options=[('1e-6', 1e-6), ('1e-8', 1e-8), ('1e-10', 1e-10)],
    value=1e-10, description='Tolerance:', layout=widgets.Layout(width='220px'))
run = widgets.Button(description='Run comparison', button_style='primary', icon='play')
notice = widgets.HTML(value='<span style="color:#92400e">Click <b>Run comparison</b> to compute results.</span>')
output = widgets.Output()

def mark_changed(change):
    if change['name'] == 'value':
        notice.value = ('<span style="color:#92400e">⚠ Parameters changed. '
                        'Click <b>Run comparison</b> to update the plots and table.</span>')

for control in (x1, x2, tolerance):
    control.observe(mark_changed, names='value')

def update(_=None):
    values = [x1.value, x2.value, tolerance.value]
    if not np.isfinite(values).all():
        notice.value = '<span style="color:#b91c1c">Enter finite starting coordinates.</span>'
        return
    run.disabled = True
    notice.value = '<span>Running comparison…</span>'
    try:
        with output:
            clear_output(wait=True)
            render_dashboard(compute_results(*values), tolerance.value)
        notice.value = '<span style="color:#166534">✓ Results match the current parameters.</span>'
    except Exception as exc:
        notice.value = ('<span style="color:#b91c1c">Error: '
                        + str(exc).replace('<', '&lt;').replace('>', '&gt;') + '</span>')
    finally:
        run.disabled = False

run.on_click(update)
display(widgets.VBox([widgets.HBox([x1, x2, tolerance]), run, notice, output]))
# No automatic run: avoids blocking Voilà startup.
